In [1]:
import pandas as pd
import os
import numpy as np
import datetime as dt
from filter_foia import filter_foia


cwd = os.getcwd()
cwd

'C:\\Users\\OITNYNWilsoS\\OneDrive - Department of Veterans Affairs\\python\\FOIA\\FOIA metrics - Tammy\\2025 Cleland-Dole Act Report'

In [2]:
#bring in va volume data to get number requests processed and increase
score_card_folder = r"C:\Users\OITNYNWilsoS\OneDrive - Department of Veterans Affairs\CRR Data Analytics (FOIA) - eFOIA\Score Card Power BI dashboards/"
va_volume = pd.read_excel(os.path.join(score_card_folder, "va volume.xlsx"))
va_volume = va_volume.rename(columns={'Increase Requests Processed':'VA Increase Requests Processed'})
va_volume = va_volume[2:]
va_volume

,Fiscal Year,VA Number of Requests Received in Fiscal Year,VA Number of Requests Processed in Fiscal Year,VA Increase Requests Processed
2,2015-01-01,29716,30436,2454.0
3,2016-01-01,34459,33313,2877.0
4,2017-01-01,24176,25237,-8076.0
5,2018-01-01,24555,22851,-2386.0
6,2019-01-01,21336,23749,898.0
7,2020-01-01,19053,19323,-4426.0
8,2021-01-01,27762,30227,10904.0
9,2022-01-01,22542,24343,-5884.0
10,2023-01-01,79590,72403,48060.0
11,2024-01-01,105725,119453,47050.0


In [3]:
#bring in backlog data from foia.gov - has an API, but not functional
python_folder = 'C:\\Users\\OITNYNWilsoS\\OneDrive - Department of Veterans Affairs\\python\\FOIA\\FOIA metrics - Tammy\\2025 Cleland-Dole Act Report'
csv = pd.read_csv(os.path.join(python_folder, "backlogs 2015 - 2028 foia-backlogs-of-foia-requests-and-administrative-appeals.csv"))
csv.head(n=10)

,Agency,Component,Fiscal Year,Number of Backlogged Requests as of End of Fiscal Year,Number of Backlogged Appeals as of End of Fiscal Year
0,Merit Systems Protection Board,Agency Overall,2016,69,2.0
1,Federal Reserve System,Agency Overall,2023,36,3.0
2,National Railroad Passenger Corporation,Agency Overall,2015,33,4.0
3,Federal Financial Institutions Examination Cou...,Agency Overall,2019,0,0.0
4,Department of the Interior,Agency Overall,2016,677,337.0
5,Federal Labor Relations Authority,Agency Overall,2020,0,0.0
6,Corporation for National and Community Service...,Agency Overall,2024,30,0.0
7,Department of Justice,Agency Overall,2024,21567,211.0
8,Federal Mediation and Conciliation Service,Agency Overall,2023,0,0.0
9,Securities and Exchange Commission,Agency Overall,2017,73,13.0


In [4]:
#use filter_foia to only extract All Agencies data
target_backlog_names = ['Fiscal Year', 'Number of Backlogged Requests as of End of Fiscal Year']
all_agencies_backlogged = filter_foia(csv, 'All agencies', target_backlog_names, 'All Agencies')
all_agencies_backlogged

,Fiscal Year,All Agencies Number of Backlogged Requests as of End of Fiscal Year
0,2015-01-01,102828
1,2016-01-01,115080
2,2017-01-01,111344
3,2018-01-01,130718
4,2019-01-01,120436
5,2020-01-01,141762
6,2021-01-01,153227
7,2022-01-01,206720
8,2023-01-01,200843
9,2024-01-01,267056


In [5]:
#pull in number of requests csv, also from foia.gov
processed_received = pd.read_csv(os.path.join(python_folder, "2024 all federal govt foia-received-processed-and-pending-foia-requests.csv"))
received_cols = ['Fiscal Year', 'Number of Requests Received in Fiscal Year']
all_agencies_received = filter_foia(processed_received, 'All agencies', received_cols, 'All Agencies')
all_agencies_received

,Fiscal Year,All Agencies Number of Requests Received in Fiscal Year
0,2015-01-01,713168
1,2016-01-01,788769
2,2017-01-01,818272
3,2018-01-01,863729
4,2019-01-01,858952
5,2020-01-01,790688
6,2021-01-01,838164
7,2022-01-01,928353
8,2023-01-01,1199699
9,2024-01-01,1501432


In [6]:
#combined backlog and received for all agencies, calculate percentage of received that are backlogs
def add_received(df1, df2, entity):
    df = pd.merge(df1, df2, on="Fiscal Year", how="inner")
    df[f"{entity} Backlog as Percentage of Requests Received"] = df[f"{entity} Number of Backlogged Requests as of End of Fiscal Year"] / df[f"{entity} Number of Requests Received in Fiscal Year"]
    return df

In [7]:
all_agencies = add_received(all_agencies_backlogged, all_agencies_received, "All Agencies")
all_agencies

,Fiscal Year,All Agencies Number of Backlogged Requests as of End of Fiscal Year,All Agencies Number of Requests Received in Fiscal Year,All Agencies Backlog as Percentage of Requests Received
0,2015-01-01,102828,713168,0.144185
1,2016-01-01,115080,788769,0.145898
2,2017-01-01,111344,818272,0.136072
3,2018-01-01,130718,863729,0.151341
4,2019-01-01,120436,858952,0.140213
5,2020-01-01,141762,790688,0.179289
6,2021-01-01,153227,838164,0.182813
7,2022-01-01,206720,928353,0.222674
8,2023-01-01,200843,1199699,0.167411
9,2024-01-01,267056,1501432,0.177868


In [8]:
#add processed data for all agencies
processed_all = pd.read_excel(os.path.join(score_card_folder, "FOIAs Processed Over Time.xlsx")) #extracted from original Score Card dashboard
processed_all = processed_all[['Fiscal Year', '"All Agencies" Federal Government']]
processed_all = processed_all.rename(columns={'"All Agencies" Federal Government': 'All Agencies Number of Requests Processed in Fiscal Year'})
processed_all

,Fiscal Year,All Agencies Number of Requests Processed in Fiscal Year
0,2015-01-01,769903.0
1,2016-01-01,759842.0
2,2017-01-01,823223.0
3,2018-01-01,830060.0
4,2019-01-01,877964.0
5,2020-01-01,772869.0
6,2021-01-01,838668.0
7,2022-01-01,878420.0
8,2023-01-01,1122211.0
9,2024-01-01,1499265.0


In [9]:
#add requests data to receive and backlogged dataframe
all_agencies = pd.merge(all_agencies, processed_all, on="Fiscal Year", how="inner")
all_agencies

,Fiscal Year,All Agencies Number of Backlogged Requests as of End of Fiscal Year,All Agencies Number of Requests Received in Fiscal Year,All Agencies Backlog as Percentage of Requests Received,All Agencies Number of Requests Processed in Fiscal Year
0,2015-01-01,102828,713168,0.144185,769903.0
1,2016-01-01,115080,788769,0.145898,759842.0
2,2017-01-01,111344,818272,0.136072,823223.0
3,2018-01-01,130718,863729,0.151341,830060.0
4,2019-01-01,120436,858952,0.140213,877964.0
5,2020-01-01,141762,790688,0.179289,772869.0
6,2021-01-01,153227,838164,0.182813,838668.0
7,2022-01-01,206720,928353,0.222674,878420.0
8,2023-01-01,200843,1199699,0.167411,1122211.0
9,2024-01-01,267056,1501432,0.177868,1499265.0


In [10]:
#do same as above for VA
agency = 'Department of Veterans Affairs'
entity = 'VA'

va_backlog = filter_foia(csv, agency, target_backlog_names, entity)
#add backlog numbers, including for OIG (+0)
va_backlog = pd.DataFrame(np.insert(va_backlog.values, 0, [dt.datetime(2025,1,1), 3549+0], axis=0), columns=va_backlog.columns)
va = pd.merge(va_volume, va_backlog, on="Fiscal Year", how="outer")

# Confirmed it is likely OIG reported number for FY 2024. So removing this code that changes 2024 data based on 2025 Annual Report numbers and OIG 2025 Annual Report numbers
#va['VA Number of Backlogged Requests as of End of Fiscal Year'] = np.where(va['Fiscal Year'] == dt.datetime(2024,1,1), 1539+2, va['VA Number of Backlogged Requests as of End of Fiscal Year'])
#va['VA Number of Requests Received in Fiscal Year'] = np.where(va['Fiscal Year'] == dt.datetime(2024,1,1), 106183+520, va['VA Number of Requests Received in Fiscal Year'])
#va['VA Number of Requests Processed in Fiscal Year'] = np.where(va['Fiscal Year'] == dt.datetime(2024,1,1), 118933+514, va['VA Number of Requests Processed in Fiscal Year'])


#calculate increase in backlog requests
va[f"{entity} Backlog as Percentage of Requests Received"] = va[f"{entity} Number of Backlogged Requests as of End of Fiscal Year"] / va[f"{entity} Number of Requests Received in Fiscal Year"]
va = va.sort_values('Fiscal Year', ascending=True)
va[f"{entity} Increase in Backlog Requests"] = va[f"{entity} Number of Backlogged Requests as of End of Fiscal Year"].diff()

va

,Fiscal Year,VA Number of Requests Received in Fiscal Year,VA Number of Requests Processed in Fiscal Year,VA Increase Requests Processed,VA Number of Backlogged Requests as of End of Fiscal Year,VA Backlog as Percentage of Requests Received,VA Increase in Backlog Requests
0,2015-01-01,29716,30436,2454.0,932,0.031364,NaN
1,2016-01-01,34459,33313,2877.0,3278,0.095128,2346
2,2017-01-01,24176,25237,-8076.0,2916,0.120615,-362
3,2018-01-01,24555,22851,-2386.0,4659,0.189737,1743
4,2019-01-01,21336,23749,898.0,2631,0.123313,-2028
5,2020-01-01,19053,19323,-4426.0,3220,0.169002,589
6,2021-01-01,27762,30227,10904.0,2166,0.07802,-1054
7,2022-01-01,22542,24343,-5884.0,824,0.036554,-1342
8,2023-01-01,79590,72403,48060.0,780,0.0098,-44
9,2024-01-01,105725,119453,47050.0,1539,0.014557,759


In [11]:
#merge all agencies and VA together
all_backlog = pd.merge(all_agencies, va, on="Fiscal Year", how="outer")
all_backlog

,Fiscal Year,All Agencies Number of Backlogged Requests as of End of Fiscal Year,All Agencies Number of Requests Received in Fiscal Year,All Agencies Backlog as Percentage of Requests Received,All Agencies Number of Requests Processed in Fiscal Year,VA Number of Requests Received in Fiscal Year,VA Number of Requests Processed in Fiscal Year,VA Increase Requests Processed,VA Number of Backlogged Requests as of End of Fiscal Year,VA Backlog as Percentage of Requests Received,VA Increase in Backlog Requests
0,2015-01-01,102828.0,713168.0,0.144185,769903.0,29716,30436,2454.0,932,0.031364,NaN
1,2016-01-01,115080.0,788769.0,0.145898,759842.0,34459,33313,2877.0,3278,0.095128,2346
2,2017-01-01,111344.0,818272.0,0.136072,823223.0,24176,25237,-8076.0,2916,0.120615,-362
3,2018-01-01,130718.0,863729.0,0.151341,830060.0,24555,22851,-2386.0,4659,0.189737,1743
4,2019-01-01,120436.0,858952.0,0.140213,877964.0,21336,23749,898.0,2631,0.123313,-2028
5,2020-01-01,141762.0,790688.0,0.179289,772869.0,19053,19323,-4426.0,3220,0.169002,589
6,2021-01-01,153227.0,838164.0,0.182813,838668.0,27762,30227,10904.0,2166,0.07802,-1054
7,2022-01-01,206720.0,928353.0,0.222674,878420.0,22542,24343,-5884.0,824,0.036554,-1342
8,2023-01-01,200843.0,1199699.0,0.167411,1122211.0,79590,72403,48060.0,780,0.0098,-44
9,2024-01-01,267056.0,1501432.0,0.177868,1499265.0,105725,119453,47050.0,1539,0.014557,759


In [12]:
#export to Excel
with pd.ExcelWriter(
    "backlog overview.xlsx",
    engine="xlsxwriter",
    date_format="YYYY-MM-DD",
    datetime_format="YYYY-MM-DD"
) as writer:
    all_backlog.to_excel(os.path.join(score_card_folder, "backlog_overview.xlsx"), sheet_name="Backlog Overview", index=False)